# Persona Lab クイックスタート

調査定義から回答生成までを、このノートブック1本で通します。

```
環境セットアップ → personas_base 構築 → 調査定義 → validate → panel → run → 結果
```

**Databricks ノートブックでもローカル Jupyter でも動きます。** 先頭で環境を判定して、
依存の入れ方とテーブルの置き場所を切り替えます。

## 所要時間の目安

| 手順 | 目安 |
|---|---|
| 依存の導入 | 1〜3分（初回のみ） |
| `personas_base` 構築（1シャード / 125,000行） | 2〜5分（初回のみ。216MB のダウンロードを含む） |
| `panel` + `run`（120人 × 3案 × 3問） | 1〜3分 |

## 前提

- Python 3.11 以上
- ローカルで動かす場合は **JDK 17 または 21**（Spark 4.0 が要求します）
- 既定では `model.endpoint: fake`（決定論的なダミー）で動くので、**推論エンドポイントが無くても最後まで通ります**。
  実際の Databricks Model Serving に繋ぐ手順は後半で説明します。

---

> 本ツールの出力はAIによるシミュレーションであり、実在する生活者の回答ではありません。
> 意思決定の根拠として単独で用いず、仮説生成・優先順位付け・調査設計の目的で使用してください。

本システムは NVIDIA が公開する Nemotron-Personas-Japan (CC BY 4.0) を使用しています。
https://huggingface.co/datasets/nvidia/Nemotron-Personas-Japan

## 1. 環境の判定

Databricks 上かどうかで、この後の分岐が決まります。

## 2. 依存の導入

- **Databricks**: `%pip install` でクラスタに入れます（インストール後にカーネルが再起動します）。
- **ローカル**: リポジトリ直下で `pip install -r requirements-pipeline.txt -r requirements-notebook.txt` を
  済ませた venv からこのノートブックを起動してください。ここでは導入済みかを確認するだけにします。

Databricks では次のセルのコメントを外して実行してください。`%pip` はセルの先頭行に置く必要があります。

In [0]:
# Databricks で実行する場合のみ、次の2行のコメントを外す。
# パスは自分のワークスペースに置いたリポジトリのものに書き換えること
# （実在するパスをこのファイルに残さない。AGENTS.md の秘密情報の扱い）
# %pip install --no-deps /Workspace/Users/<自分のユーザー名>/concept_digitaltwin
# dbutils.library.restartPython()


In [0]:
import os
import sys
from pathlib import Path

ON_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

print("実行環境      :", "Databricks" if ON_DATABRICKS else "ローカル")
print("Python        :", sys.version.split()[0])

In [0]:
# 依存が揃っているかの確認（足りなければここで気づけるようにする）
missing = []
for module in ("pyspark", "delta", "yaml", "huggingface_hub", "pandas"):
    try:
        __import__(module)
    except ImportError:
        missing.append(module)

if missing:
    raise SystemExit(
        f"依存が足りません: {', '.join(missing)}\n"
        "ローカルなら `pip install -r requirements-pipeline.txt -r requirements-notebook.txt` を実行してください。"
    )

# リポジトリを clone しただけで使う場合は、パッケージの場所を通しておく。
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if (repo_root / "persona_sim").is_dir() and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import persona_sim

print("persona_sim", persona_sim.__version__)

## 3. テーブルの置き場所と出力先

**接続情報はコードにも調査定義にも書きません。** 環境変数から解決します。

| 環境変数 | 用途 |
|---|---|
| `PERSONA_SIM_CATALOG` / `PERSONA_SIM_SCHEMA` | Unity Catalog のカタログ／スキーマ。両方揃うとマネージドテーブルを使う |
| `PERSONA_SIM_WAREHOUSE` | テーブルを置くベースパス。カタログ未設定のときに使う |
| `PERSONA_SIM_OUTPUT_DIR` | `run_metadata.json` などファイル成果物の出力先 |

Databricks では Unity Catalog を、ローカルでは作業ディレクトリ配下を既定にします。
**Databricks で使う場合は、下のカタログ名とスキーマ名を自分の環境のものに置き換えてください。**

In [0]:
if ON_DATABRICKS:
    # ここは自分のワークスペースの値に置き換える。
    # 実在するカタログ名・パスをこのファイルに残さないこと（AGENTS.md の秘密情報の扱い）
    os.environ["PERSONA_SIM_CATALOG"] = "research_system"
    os.environ["PERSONA_SIM_SCHEMA"] = "default"
    os.environ["PERSONA_SIM_OUTPUT_DIR"] = "/Workspace/Users/yosuke_kazama@kirin.co.jp/デジタルツイン"
else:
    workdir = Path.cwd() / "_quickstart"
    os.environ.setdefault("PERSONA_SIM_WAREHOUSE", str(workdir / "warehouse"))
    os.environ.setdefault("PERSONA_SIM_OUTPUT_DIR", str(workdir / "outputs"))

from persona_sim.config import output_dir, storage_config

storage = storage_config()
print("テーブル配置 :", "Unity Catalog" if storage.uses_catalog else storage.warehouse)
print("出力先       :", output_dir())


## 4. SparkSession

Databricks では既に張られているセッションをそのまま使います。ローカルでは Delta 拡張を
有効にしたセッションを作ります（初回は Maven から Delta の JAR を取得するので少し待ちます）。

In [0]:
from persona_sim.spark import get_spark

spark = get_spark("persona-lab-quickstart")
print("Spark", spark.version)

## 5. `personas_base` を作る（M1）

Nemotron-Personas-Japan をリビジョン固定で取り込み、正規化して Delta に書きます。

- `occupation` を「業種 / 規模 / 役職 / 就業状態」に分解します
  （規則は `docs/schema/occupation-parsing.md`。実データ125,000行で検証済み）
- `age_band_5` / `age_band_10` を派生させます
- `source_version` にリビジョンを残します。**あとで結果を突き合わせるときに必要です**

1シャード（125,000行）だけ取り込みます。全量が必要なら `shards=8` にしてください。

> Hugging Face に到達できない環境では、あらかじめ落とした parquet のパスを
> `local_paths=["/path/to/train-00000-of-00008.parquet"]` で渡せます。

In [0]:
import os

import pandas as pd

from persona_sim.config import storage_config
from persona_sim.personas.build import normalize_personas
from persona_sim.personas.source import DEFAULT_REVISION, download_shards, source_version
from persona_sim.storage import delta
from persona_sim.storage.locator import PERSONAS_BASE, locator

# cell 8 の setdefault は既存環境変数を上書きしないため再設定
storage = storage_config()
personas_locator = locator(PERSONAS_BASE, storage)

if delta.table_exists(spark, personas_locator):
    print("既に構築済みなのでスキップします:", personas_locator.describe())
else:
    # Serverless では Spark がローカルファイルにアクセスできないため
    # pandas で読んでから Spark DataFrame に変換する
    local_paths = download_shards(shards=1)
    pdf = pd.concat([pd.read_parquet(p) for p in local_paths], ignore_index=True)
    raw = spark.createDataFrame(pdf)

    version = source_version(DEFAULT_REVISION)
    normalized = normalize_personas(raw, version)
    delta.write_table(normalized, personas_locator, mode="overwrite")

    rows = delta.read_table(spark, personas_locator).count()
    tbl_version = delta.table_version(spark, personas_locator)
    print("行数           :", f"{rows:,}")
    print("Delta バージョン:", tbl_version)
    print("source_version :", version)

In [0]:
personas = delta.read_table(spark, personas_locator)
print("列数:", len(personas.columns))
personas.select(
    "uuid", "sex", "age", "age_band_10", "prefecture",
    "occupation_industry", "occupation_scale", "employment_status",
).show(5, truncate=False)

## 6. 調査定義（dict で渡す）

調査定義はこのシステムへの唯一の入力です。YAML でも dict でも同じものを渡せます
（`SPEC_PHASE1.md` §10.2）。ノートブックでは dict が扱いやすいのでこちらを使います。

### 提示設計は3軸で指定します

| 軸 | フィールド | 意味 |
|---|---|---|
| 誰が何を見るか | `sample_overlap` | `disjoint`（1人1案） / `allow_overlap` / `same`（全員が全案） |
| どう見せるか | `presentation` | `sequential`（1件ずつ） / `simultaneous`（同時比較） |
| 覚えているか | `memory` | `none` / `within_stimulus` / `full_session` |

既定の **`same` × `sequential` × `none`** は「反実仮想モナディック」です。全員が全案を
互いに独立したセッションで評価するので、サンプル差による交絡が無く、順序効果も発生しません。
実査では取れない設計です。その代わりセッション数がコンセプト数倍になります。

### 対象者条件（スクリーナー）

実務のコンセプト調査は、ほぼ必ず対象者条件（カテゴリ利用者・購入頻度など）を持ちます。
`panel.screener` に書けば調査定義の一部になり、手順9の `screen` が判定します。

LLM ペルソナでは条件の満たし方が **3通り**あり、**費用と、得られるものが違います**。

| | `ask`（実際に聞く） | `assume`（前提として与える） | `infer`（蓋然性で選ぶ） |
|---|---|---|---|
| やること | スクリーナー設問に答えさせ、通過者だけを残す | 聞かずに「あなたは月1回以上ビールを飲みます」を全候補に付与する | 別の LLM に候補をまとめて見せ、条件に合いそうな人にだけ付与する |
| 費用 | `panel.size × oversample_factor × 条件数` セッション | **0** | `候補数 ÷ batch_size` 回 |
| インシデンス | **実測できる**（実査と突き合わせられる） | **測れない**。`null` として記録します | **推定値**。実測とは別の欄に記録します |
| パネルの構成 | 割り付けセル内で条件該当者に偏る | 割り付けどおりのまま、行動だけを付与する | `ask` と同じく条件該当者に偏る |
| 内的整合性 | ペルソナ本人の記述と矛盾しない | **ナラティブと矛盾しうる** | ナラティブを読んで選ぶので矛盾しにくい |

**どれが正しいというものではなく、答えている問いが違います。** `ask` は「条件該当者は
どう反応するか」、`assume` は「割り付けどおりの構成の人が、その行動を持つとしたら
どう反応するか」、`infer` は「条件に当てはまりそうな人が、その行動を持つとしたら
どう反応するか」です。ここではインシデンスを実測できる `ask` を使います。`mode` を
変えるだけで切り替わります。

**方式ごとに書ける形が違います。** `ask` は本人に選択肢を見せて答えさせるので
`questions`（設問文・選択肢・`pass_if`）が要ります。`assume` と `infer` は選択肢を
提示しないので、`conditions` に対象者条件を自然言語で書くだけです。

```python
# 聞かない。費用0
"screener": {"mode": "assume", "conditions": ["缶チューハイを月1回以上飲む"]}

# 候補をまとめて判定させる。ask より安く、assume のような全員への強制も避けられる
"screener": {
    "mode": "infer",
    "oversample_factor": 2,
    "conditions": ["缶チューハイを月1回以上飲む"],
    "infer": {
        "batch_size": 20,                 # 1回の判定に渡すペルソナ数
        "model": {"max_tokens": 256},     # 省略したキーは model をそのまま引き継ぐ
        "include_summary": True,          # 判定に見せる情報はここで絞れる
    },
}
```

`infer` の通過率は **推定値** です。本人には聞いていないので `ask` の実測値と同じもの
として読まないでください。判定結果には `inferred` フラグが立ちます。

どの方式でも、確定したペルソナのカード末尾に前提ブロックが付きます。
`memory: none` では設問ごとに独立したセッションになるため、会話履歴では前提を
引き継げないからです。見出しで由来が分かるようにしてあります。

```
【調査前の確認】                        ← ask: 本人が実際に選んだ選択肢
- 缶チューハイの飲用頻度: 月2〜3回

【前提】                                ← assume: 調査定義に書いた premise
- 缶チューハイ・缶ハイボールを月1回以上飲む

【推定前提】                            ← infer: 判定で付与した条件
- 缶チューハイ・缶ハイボールを月1回以上飲む
```


In [0]:
OPTIONS_5 = [
    "ぜひ購入したい",
    "やや購入したい",
    "どちらともいえない",
    "あまり購入したくない",
    "まったく購入したくない",
]

QUESTIONS = [
    {
        "id": "q_intent",
        "text": "この商品を購入したいと思いますか。",
        "type": "single",
        "options": OPTIONS_5,
        "randomize_options": False,  # 順序尺度はシャッフルしない
        "top_box": [1, 2],
    },
    {
        "id": "q_novelty",
        "text": "この商品は新しいと思いますか。",
        "type": "single",
        "options": [
            "とても新しい",
            "やや新しい",
            "どちらともいえない",
            "あまり新しくない",
            "まったく新しくない",
        ],
        "randomize_options": False,
        "top_box": [1, 2],
    },
    {
        "id": "q_reason",
        "text": "そう思った理由を教えてください。",
        "type": "open",
        "max_length": 150,
    },
]

survey_dict = {
    "survey": {"id": "quickstart_rtd", "name": "RTD新コンセプト評価（クイックスタート）", "type": "concept"},
    "panel": {
        "size": 10,
        "seed": 42,
        "quotas": {
            "mode": "count",
            "cells": [
                {"cell_id": "M_20_40s", "sex": "男", "age_min": 20, "age_max": 49, "n": 5},
                {"cell_id": "F_20_40s", "sex": "女", "age_min": 20, "age_max": 49, "n": 5},
            ],
        },
        # 酒類カテゴリなので未成年を含めない
        "filters": {"age_min": 20},
    },
    # 対象者条件。"assume" にすると聞かずに前提として与えます（費用0・インシデンスは測れません）
    "screening": {
        "mode": "ask",
        "oversample_factor": 3,
        "questions": [
            {
                "id": "sc1",
                "text": "缶チューハイ・缶ハイボールなどをどのくらいの頻度で飲みますか。",
                "label": "缶チューハイの飲用頻度",
                "type": "single",
                "options": ["週2回以上", "週1回", "月2〜3回", "月1回", "それ以下・飲まない"],
                "pass_if": [1, 2, 3, 4],
                "premise": "缶チューハイ・缶ハイボールを月1回以上飲む",
            }
        ],
    },
    "stimuli": [
        {
            "id": "c1",
            "name": "ゆずハイボール",
            "text": "【商品名】ゆずハイボール\n【特徴】国産ゆず果汁を使った、甘さ控えめの缶ハイボール。アルコール5%。\n【価格】希望小売価格 205円（350ml）",
        },
        {
            "id": "c2",
            "name": "無糖レモンサワー ストロング",
            "text": "【商品名】無糖レモンサワー ストロング\n【特徴】糖類・甘味料ゼロ、レモン果汁10%。アルコール9%。\n【価格】希望小売価格 190円（350ml）",
        },
        {
            "id": "c3",
            "name": "クラフトジンソーダ 山椒",
            "text": "【商品名】クラフトジンソーダ 山椒\n【特徴】和素材のクラフトジンを使った、山椒が香るジンソーダ。アルコール7%。\n【価格】希望小売価格 260円（350ml）",
        },
    ],
    # 設問は「そのペルソナが何番目に見るコンセプトについて聞くか」（slot）に紐づくので、
    # 評価するコンセプトの数だけ展開します（下の内包表記が QUESTIONS から機械的に作ります）。
    # measure はコンセプト横断で同じ問いとして束ねるキーで、比較表はこれで組まれます。
    # 設問間で記憶を持たせたいときは "remember": ["q_intent_1"] のように書きます
    # （ここでは全案を独立したセッションで評価するので持たせていません）。
    "questions": [
        {**question, "id": f"{question['id']}_{slot}", "slot": slot, "measure": question["id"]}
        for slot in (1, 2, 3)
        for question in QUESTIONS
    ],
    # 本調査（回答生成）の設定。モデル・プロンプト・ペルソナカードをまとめて持ちます。
    # スクリーニング側の設定は上の "screening" にあります。
    "main_survey": {
        "model": {
            # fake = 決定論的なダミー。実エンドポイント無しで最後まで通せる。
            # Databricks に繋ぐときは次の小見出しを見て endpoint と deployment を書き換える
            # （エンドポイント名はコミットに残さない。AGENTS.md の秘密情報の扱い）
            "endpoint": "fake",
            "deployment": "quickstart",
            "thinking": False,  # 常に false（AGENTS.md 不変条件）
            "max_tokens": 500,
            "concurrency": 8,
        },
        "prompt": {
            # [system]・見出し・回答指示文をここで差し替えられます（省略時は既定のまま）。
            # 例として単一回答の指示行だけを上書きしています。
            "rules": {"single": "当てはまる番号を1つだけ答えてください。"},
        },
        "persona_card": {
            # ペルソナカードに載せるナラティブ列。カテゴリに応じて差し替えられます
            # （スポーツ用品なら culinary_persona → sports_persona など）。
            "persona_fields": [
                {"field": "cultural_background", "label": "生活背景"},
                {"field": "culinary_persona", "label": "食まわり"},
            ],
            # 属性行は attributes で属性ごとに選べます（省略時は既定の6項目）。
            # 総括文を外すなら "include_summary": False。
        },
    },
    "output": {
        "segments": ["total", "sex", "age_band_10"],
        "formats": ["csv", "xlsx"],
    },
}

from persona_sim.panel.loader import survey_from_dict

survey = survey_from_dict(survey_dict)
print("調査:", survey.survey_id, "/ コンセプト", survey.stimuli_count, "件 / 設問", len(survey.questions), "問")


### 実際の Databricks Model Serving に繋ぐには

上の調査定義の `model` を次のように変えるだけです。

```python
"model": {
    "endpoint": "databricks",
    "deployment": "databricks-gemini-3-5-flash-lite",
    "thinking": False,
    "concurrency": 32,
    "structured_output": "auto",
}
```

認証は `databricks-sdk` の既定の認証チェーンに任せます。Databricks 上なら環境から解決され、
ローカルからは `DATABRICKS_HOST` / `DATABRICKS_TOKEN` を読みます。**トークンをノートブックに
書かないでください。**

> 最初に本物のエンドポイントへ繋ぐときは、`panel.size` を 10 程度まで落として試してください。
> レスポンス形式とトークン計上を確認してから本番規模に上げるほうが安全です。

In [0]:
# OPTIONS_5 = [
#     "ぜひ購入したい",
#     "やや購入したい",
#     "どちらともいえない",
#     "あまり購入したくない",
#     "まったく購入したくない",
# ]

# QUESTIONS = [
#     {
#         "id": "q_intent",
#         "text": "この商品を購入したいと思いますか。",
#         "type": "single",
#         "options": OPTIONS_5,
#         "randomize_options": False,  # 順序尺度はシャッフルしない
#         "top_box": [1, 2],
#     },
#     {
#         "id": "q_novelty",
#         "text": "この商品は新しいと思いますか。",
#         "type": "single",
#         "options": [
#             "とても新しい",
#             "やや新しい",
#             "どちらともいえない",
#             "あまり新しくない",
#             "まったく新しくない",
#         ],
#         "randomize_options": False,
#         "top_box": [1, 2],
#     },
#     {
#         "id": "q_reason",
#         "text": "そう思った理由を教えてください。",
#         "type": "open",
#         "max_length": 150,
#     },
# ]
#
# survey_dict = {
#     "survey": {"id": "quickstart_rtd_full", "name": "RTD新コンセプト評価（本番規模）", "type": "concept"},
#     "panel": {
#         "size": 120,
#         "seed": 42,
#         "quotas": {
#             "mode": "count",
#             "cells": [
#                 {"cell_id": "M_20s", "sex": "男", "age_min": 20, "age_max": 29, "n": 20},
#                 {"cell_id": "M_30s", "sex": "男", "age_min": 30, "age_max": 39, "n": 20},
#                 {"cell_id": "M_40s", "sex": "男", "age_min": 40, "age_max": 49, "n": 20},
#                 {"cell_id": "F_20s", "sex": "女", "age_min": 20, "age_max": 29, "n": 20},
#                 {"cell_id": "F_30s", "sex": "女", "age_min": 30, "age_max": 39, "n": 20},
#                 {"cell_id": "F_40s", "sex": "女", "age_min": 40, "age_max": 49, "n": 20},
#             ],
#         },
#         # 酒類カテゴリなので未成年を含めない
#         "filters": {"age_min": 20},
#     },
#     # 手順6の本体と同じスクリーナー。規模を上げるとここの費用も比例して増える
#     "screening": {
#         "mode": "ask",
#         "oversample_factor": 3,
#         "questions": [
#             {
#                 "id": "sc1",
#                 "text": "缶チューハイ・缶ハイボールなどをどのくらいの頻度で飲みますか。",
#                 "label": "缶チューハイの飲用頻度",
#                 "type": "single",
#                 "options": ["週2回以上", "週1回", "月2〜3回", "月1回", "それ以下・飲まない"],
#                 "pass_if": [1, 2, 3, 4],
#                 "premise": "缶チューハイ・缶ハイボールを月1回以上飲む",
#             }
#         ],
#     },
#     "stimuli": [
#         {
#             "id": "c1",
#             "name": "ゆずハイボール",
#             "text": "【商品名】ゆずハイボール\n【特徴】国産ゆず果汁を使った、甘さ控えめの缶ハイボール。アルコール5%。\n【価格】希望小売価格 205円（350ml）",
#         },
#         {
#             "id": "c2",
#             "name": "無糖レモンサワー ストロング",
#             "text": "【商品名】無糖レモンサワー ストロング\n【特徴】糖類・甘味料ゼロ、レモン果汁10%。アルコール9%。\n【価格】希望小売価格 190円（350ml）",
#         },
#         {
#             "id": "c3",
#             "name": "クラフトジンソーダ 山椒",
#             "text": "【商品名】クラフトジンソーダ 山椒\n【特徴】和素材のクラフトジンを使った、山椒が香るジンソーダ。アルコール7%。\n【価格】希望小売価格 260円（350ml）",
#         },
#     ],
#     # 設問は「そのペルソナが何番目に見るコンセプトについて聞くか」（slot）に紐づくので、
#     # 評価するコンセプトの数だけ展開します（下の内包表記が QUESTIONS から機械的に作ります）。
#     # measure はコンセプト横断で同じ問いとして束ねるキーで、比較表はこれで組まれます。
#     # 設問間で記憶を持たせたいときは "remember": ["q_intent_1"] のように書きます
#     # （ここでは全案を独立したセッションで評価するので持たせていません）。
#     "questions": [
#         {**question, "id": f"{question['id']}_{slot}", "slot": slot, "measure": question["id"]}
#         for slot in (1, 2, 3)
#         for question in QUESTIONS
#     ],
#     "main_survey": {
#         "model": {
#             # fake = 決定論的なダミー。実エンドポイント無しで最後まで通せる。
#             "endpoint": "fake",
#             "deployment": "quickstart",
#             "thinking": False,  # 常に false（AGENTS.md 不変条件）
#             "concurrency": 8,
#         },
#         "prompt": {
#             # 例として単一回答の指示行だけを上書きしています。
#             "rules": {"single": "当てはまる番号を1つだけ答えてください。"},
#         },
#         "persona_card": {
#             "persona_fields": [
#                 {"field": "cultural_background", "label": "生活背景"},
#                 {"field": "culinary_persona", "label": "食まわり"},
#             ],
#         },
#     },
#     "output": {"segments": ["total", "sex", "age_band_10"], "formats": ["csv"]},
# }

# from persona_sim.panel.loader import survey_from_dict

# survey = survey_from_dict(survey_dict)
# print("調査:", survey.survey_id, "/ コンセプト", survey.stimuli_count, "件 / 設問", len(survey.questions), "問")

## 7. `validate` — 実行前に止められるものを止める

調査定義の矛盾はここで全部出します（1つ直すたびに再実行しなくて済むよう、まとめて返します）。

**セッション数の見積もりを必ず表示します。** 既定の `same` ではコンセプト数倍になり、
コストに直結するので、実行前に目に入るようにしています。

スクリーニングのセッション数も別行で出ます。`assume` を選んだ場合は「`ask` ならいくらか」を
併記します。費用とのバーターで方式を選べるようにするためです。

In [0]:
from persona_sim.panel.validate import validate_static

report = validate_static(survey)

for issue in report.errors:
    print("エラー", issue)
for issue in report.warnings:
    print("警告  ", issue)

if report.estimate:
    print()
    for line in report.estimate.lines():
        print(line)

assert report.ok, "調査定義にエラーがあります"

## 8. `panel` — 割り付けどおりに人を選ぶ（M2）

- 抽出は**乱数を使いません**。`sha2(uuid | seed | cell_id)` の昇順で取るので、
  パーティション数や並列度が変わっても同じ人が選ばれます
- セル条件が重なっても、同一ペルソナがパネルに2回入ることはありません
- `assigned_stimuli` に「誰が何をどの順で見るか」が入ります（**パネルが確定してから**埋まります）

スクリーナーが `ask` のとき、ここで引くのは**候補**です（`panel.size × oversample_factor` 人、`role = candidate`）。
誰が本調査に進むかは次の手順9で決まります。

In [0]:
from persona_sim.panel.build import build_panel, composition_lines

panel_result = build_panel(spark, survey, storage)

for line in composition_lines(survey, panel_result.selection):
    print(line)

In [0]:
panel_result.panel.select("persona_uuid", "cell_id", "assigned_stimuli", "weight").show(5, truncate=False)

## 9. `screen` — 判定してパネルを確定する（M4）

手順8で引いたのは候補です。ここで通過者を確定パネル（`main`）に、超過分を `reserve` に、
非通過を `screened_out` に振り分けます。

```
validate → panel（候補を引く・LLM なし）→ screen（判定してパネル確定）→ run
```

`assume` とスクリーナー無しでは「実行不要」と表示して何もしません（`infer` は判定するので
`ask` と同じくここで動きます）。**この4段はいつでも
そのまま書けます。** 対象者条件の有無でノートブックを分ける必要はありません。

通過者が足りなければ `oversample_factor` を2倍にして最大3回まで候補を広げます。抽出は
ハッシュ順なので、広げても既存の候補はそのまま残り、判定済みの相手に聞き直しません。

In [0]:
from persona_sim.panel.screening import screen_survey

screening = screen_survey(spark, survey, storage)

print(f"方式            : {screening.method}")
print(f"セッション      : {screening.sessions_ok} / {screening.sessions_total}")
print(f"オーバーサンプル: {screening.oversample_actual} 倍（試行 {screening.attempts} 回）")
print()
print(f"{'セル':<12}{'確定':>8}{'予備':>8}{'非通過':>10}{'通過率':>10}")
for cell in survey.panel.quotas.cells:
    cell_id = cell.cell_id
    rate = screening.incidence.get(cell_id)
    print(
        f"{cell_id:<12}{screening.achieved.get(cell_id, 0):>8}"
        f"{screening.reserve.get(cell_id, 0):>8}{screening.screened_out.get(cell_id, 0):>10}"
        f"{(f'{rate:.1%}' if rate is not None else '-'):>10}"
    )
print()
print(f"全体のインシデンス: {screening.incidence['total']:.1%}")

非通過のレコードは**削除しません**。インシデンスの検証に使うためです（`AGENTS.md` の不変条件）。

In [0]:
from pyspark.sql import functions as F

from persona_sim.panel.screening import load_premises
from persona_sim.storage.locator import PANELS

# 非通過者は削除せず残ります（インシデンスの検証に使うため）
panels_after = delta.read_table(spark, locator(PANELS, storage)).filter(
    F.col("survey_id") == F.lit(survey.survey_id)
)
panels_after.groupBy("role").count().orderBy("role").show()

# 通過者のカードに載る前提を確認する
main_uuids = [
    row["persona_uuid"]
    for row in panels_after.filter("role = 'main'").select("persona_uuid").limit(3).collect()
]
for uuid, premise in load_premises(spark, survey, storage, main_uuids).items():
    print(uuid, "→", premise.render().replace("\n", " "))

# 割り当ては確定パネルにだけ入ります（手順8の時点では候補なのでまだ空でした）
panels_after.filter("role = 'main'").select(
    "persona_uuid", "cell_id", "assigned_stimuli", "weight"
).show(5, truncate=False)

## 10. `run` — 回答を生成する（M3 / M6）

`design.memory` が「1セッションの範囲」を決めます。ここでは `none` なので、設問1件ごとに
独立したセッションになります。

**中断しても、同じセルをもう一度実行すれば続きから再開します。** 完了済みのセッションは
スキップされ、`responses` の行が二重になることもありません。

本調査に進むのは手順9で確定した `role = main` だけです。非通過者（`screened_out`）には聞きません。

In [0]:
from persona_sim.run.run import run_survey

run_result = run_survey(spark, survey, storage)

print("セッション        :", run_result.sessions_ok, "/", run_result.sessions_total)
print("スキップ（再開）  :", run_result.sessions_skipped)
print("失敗              :", run_result.sessions_failed)
print("書き出したレコード:", f"{run_result.records_written:,}")
print("トークン          :", f"入力 {run_result.input_tokens:,} / 出力 {run_result.output_tokens:,}")
print("品質フラグ        :", run_result.flag_counts)

for warning in run_result.warnings:
    print("警告", warning)

## 11. 生データを見る

`responses` はモデルが**実際に見たとおり**に保存します。集計しやすく整形したものではありません。

- `answer_codes` は**提示順**での選択肢番号の並びです（single/scale は1要素、multi は選んだぶん全部）
- `options_order` は「定義順の番号」を提示順に並べたもの。定義順に戻すには
  `options_order[code - 1]` を引きます（シャッフルしていなければ両者は一致します）
- `flags` は立てるだけで、**行を除外しません**。集計時に含める／除くを選べるようにするためです

In [0]:
from pyspark.sql import functions as F

from persona_sim.storage import delta
from persona_sim.storage.locator import RESPONSES, locator

responses = delta.read_table(spark, locator(RESPONSES, storage)).filter(
    F.col("survey_id") == F.lit(survey.survey_id)
)
print("レコード数:", responses.count())

# 回答したのは確定パネルだけ（非通過者には聞いていない）
print("回答したペルソナ:", responses.select("persona_uuid").distinct().count(),
      f"（確定パネル {survey.panel.size} 人）")

responses.select(
    "persona_uuid", "stimulus_id", "question_id", "sequence",
    "answer_raw", "answer_codes", "options_order", "flags", "attempt",
).show(5, truncate=False)

In [0]:
# 自由回答はそのまま読めます
responses.filter("question_id = 'q_reason'").select("persona_uuid", "stimulus_id", "answer_text").show(
    5, truncate=False
)

## 12. 集計する（M5）

`aggregate_survey()` が `SPEC_PHASE1.md` §7 のクロス集計表・コンセプト比較表を作り、
§7.4 の出力一式（CSV と `report.xlsx`）を書き出します。

読むときの前提:

- **`n` はウェイト適用前の実数、`%` はウェイト適用後**（ウェイトが全て 1.0 なら一致します）
- **平均**は選択肢番号を逆順スコア化した値（5段階なら 1→5点）。**順序尺度の設問だけ**出します
- 提示順の番号は `options_order` で**定義順に戻してから**集計しています。この読み替えを
  飛ばすと、選択肢をシャッフルした設問で結果の意味が変わります
- **品質フラグの立った回答も除外していません**（§8）。除いた場合の n を併記しています

出力に統計的推測の語彙（p値・有意差・信頼区間）は使いません（`AGENTS.md` 禁止事項）。
コンセプト間の**相対比較**として読んでください。

In [0]:
from persona_sim.aggregate.aggregate import aggregate_survey

aggregate_result = aggregate_survey(spark, survey, storage, output_dir())

print("集計対象の回答:", aggregate_result.answers, "件")
print()
for path in aggregate_result.written:
    print(" ", path)
for note in aggregate_result.notes:
    print("注記:", note)

### コンセプト比較表（§7.2）

コンセプトを表側、指標を表頭にした横持ち表です。

In [0]:
import pandas as pd

from persona_sim.aggregate.tables import stacked_crosstab_table


def as_frame(table) -> pd.DataFrame:
    return pd.DataFrame(table.rows, columns=list(table.columns))


# 全設問 × 全コンセプトを1枚に積んだ表。コンセプトを measure 横断で見比べるのはこちら。
summary = stacked_crosstab_table(
    survey, aggregate_result.crosstabs, key="crosstab_all", title=""
)
display(as_frame(summary))


## 13. セグメント別に見る

`output.segments` に書いた軸がすべてクロス集計表の表側に展開されます。
`total` / 属性列（`sex` など）/ `_x_` でつないだ合成軸（`sex_x_age_band_10`）が使えます。

In [0]:
from persona_sim.aggregate.tables import concept_axis_table

# 1つの measure について、コンセプトを表側・選択肢を表頭に置いた表（ラベルつき）。
intent = concept_axis_table(
    survey,
    aggregate_result.crosstabs,
    "q_intent",
    key="crosstab_q_intent",
    title="購入意向",
    with_segment=True,
)
print(f"■ {intent.title}")
display(as_frame(intent))


## 14. 実行メタデータ

実査と突き合わせるために必要な情報を `runs` テーブルと `run_metadata.json` に残します
（`SPEC_PHASE1.md` §9）。

**再現性の範囲に注意してください。** `seed` と各バージョンが揃えば、誰にどのコンセプトを
どの順・どの選択肢順・どのプロンプトで提示したかまでは再現されます。
**生成結果そのものの一致は保証しません**（エンドポイント側の非決定性は除去できません）。
だから `answer_raw` / `attempt` / `flags` を生データに残しています。

In [0]:
import json

from persona_sim.metadata import write_metadata

metadata_path = write_metadata(spark, survey, run_result, storage, output_dir())
metadata = json.loads(Path(metadata_path).read_text(encoding="utf-8"))

print("出力先:", metadata_path)
print()
print("data_versions   :", metadata["data_versions"])
print("design          :", metadata["design"])
print("panel achieved  :", metadata["panel"]["achieved"])
print("flags           :", metadata["execution"]["flags"], f"（{metadata['execution']['flags_scope']} 単位）")
print("tokens          :", metadata["execution"]["tokens"])
print("再現性の範囲    :", metadata["reproducibility"]["guaranteed"])

## 15. 実際に送ったプロンプトを確認する

設定値だけでは「結局どう聞いたのか」が読み取れません。実行後に **1ペルソナ分だけ**、
実際にエンドポイントへ送ったメッセージ列を `prompt_sample.md` に残しています
（`SPEC_PHASE1.md` §9）。**その1名については全設問**が、聞いた順に並びます。

記憶を持つ設問（`questions[].remember`）では、先行設問の Q&A を再生してから聞くので、
`[system]` / `[user]` / `[assistant]` / `[user]` … という列になります。再生される
`[assistant]` は、そのペルソナが実際に返した回答です。

記録するペルソナはキー順で決まるので、**同じ調査を何度実行しても同じ1名**です。
上の `prompt` で見出しや指示行を変えると、ここの内容もそのまま変わります。


In [0]:
from IPython.display import Markdown, display

prompt_sample_path = Path(metadata_path).parent / "prompt_sample.md"
display(Markdown(prompt_sample_path.read_text(encoding="utf-8")))


## 16. 再開の確認

もう一度 `run` を実行すると、完了済みのセッションはすべてスキップされ、
`responses` の行数も変わりません。中断した調査を同じコマンドで再開できるのはこの仕組みです。

In [0]:
before = responses.count()
again = run_survey(spark, survey, storage)

print("スキップ:", again.sessions_skipped, "/", again.sessions_total)
print("追記    :", again.records_written)
print("行数    :", before, "→", delta.read_table(spark, locator(RESPONSES, storage)).filter(
    F.col("survey_id") == F.lit(survey.survey_id)
).count())

## 次にやること

- **本物のエンドポイントに繋ぐ**: 手順6の `model` を `databricks` に変え、まず10人規模で試す
- **規模を上げる**: `panel.size` と `stimuli` を増やす。`validate` の見積もりでセッション数を確認してから
- **提示設計を変える**: `sample_overlap` を `disjoint`（1人1案）にすると実査の monadic と同じになります。
  有効サンプル数が `panel.size ÷ コンセプト数` に減る点に注意してください

- **出力一式を配る**: 手順12が `outputs/{survey_id}/` に CSV と `report.xlsx` を書きます。
  集計をやり直さずファイルだけ作り直すなら `persona-sim export survey.yaml --xlsx` です

未実装のもの:

| | |
|---|---|
| M7 スループット調整 | 大規模調査向けの並列度の詰め |

---

本システムは NVIDIA が公開する Nemotron-Personas-Japan (CC BY 4.0) を使用しています。
https://huggingface.co/datasets/nvidia/Nemotron-Personas-Japan